# 새 모델(NAD) 실행 노트북 — Moving-Camera AV-SELD

위에서부터 **순서대로** 실행합니다. 바꿀 곳은 **2번 설정 칸**뿐입니다. 런타임은 **GPU**(A100 권장, 고용량 RAM)로 하세요.

**세션이 끊기면**: 1, 2번 셀을 다시 실행하고 → 3, 4번(로컬 준비, 이미 있으면 건너뜀)을 다시 실행한 뒤 → 끊긴 학습 셀을 다시 실행하면 **마지막 에폭 다음부터 이어 합니다**
(에폭 = `steps_per_epoch`번 갱신. 에폭마다 `last.pth`, `train_log.csv`, `train_out.log`를 드라이브의 `runs/<실행 이름>/`에 저장).

| 저장 위치 | 내용 | 크기(예상) |
|---|---|---|
| 코랩 로컬 `/content/nad` | 원본 FOA(train 방), 미리 렌더링 오디오 캐시 | 약 3GB + 약 18GB (세션마다 다시 만듦) |
| 드라이브 `New_Architecture_dev/cache` | 라벨 캐시, CLAP 캐시, OWL-ViT 캐시 | 수십 MB + 약 0.1GB + **약 11~12GB** (7번 전에 다시 확인) |
| 드라이브 `New_Architecture_dev/runs` | 체크포인트, 로그, 예측 | 실행당 약 0.2GB |

`for_dataset/` 안의 파일은 읽기만 합니다(코드는 `__pycache__`도 만들지 않게 `python -B`로 실행).

## 1. 드라이브 연결, 패키지

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks/ASR_2026-2"   #@param {type:"string"}
NAD_DIR = f"{DRIVE_ROOT}/New_Architecture_dev"
import os
os.chdir(NAD_DIR)
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
!pip install -q librosa soundfile pyyaml tqdm
import transformers, huggingface_hub; print("transformers", transformers.__version__, "| huggingface_hub", huggingface_hub.__version__)
!nvidia-smi -L
!free -g | head -2
!df -h /content | tail -1
!df -h /content/drive | tail -1

## 2. 설정 칸
- `seed`: 시드는 1번만 돌립니다.
- `total_steps`: 총 갱신 횟수. **N1-pre와 N1-otf는 같은 값**으로 둡니다(1단계 비교 조건).
- `steps_per_epoch`: 이만큼 갱신할 때마다 val 채점과 체크포인트 저장(이어 하기 단위).
- `stage1_winner`: 6번 비교 뒤, val로 더 나은 1단계 실행 이름을 적습니다(2단계 초기값).

In [ ]:
seed = 1                   #@param {type:"integer"}
batch_size = 32            #@param {type:"integer"}
total_steps = 20000        #@param {type:"integer"}
steps_per_epoch = 500      #@param {type:"integer"}
lr = 0.001                 #@param {type:"number"}
num_workers = 6            #@param {type:"integer"}
train_stages = "0,1"       #@param ["0,1", "0"]
stage1_winner = ""         #@param {type:"string"}
n2_total_steps = 20000     #@param {type:"integer"}
finetune_after_step = 0    #@param {type:"integer"}

def ARGS(preset, **extra):
    sets = dict(drive_root=DRIVE_ROOT, seed=seed, batch_size=batch_size, total_steps=total_steps, steps_per_epoch=steps_per_epoch,
                lr=lr, num_workers=num_workers, train_stages=train_stages)
    sets.update(extra)
    return f'--preset {preset} ' + ' '.join(f'--set {k}="{v}"' for k, v in sets.items())

def run_name(preset):
    import sys; sys.path.insert(0, NAD_DIR)
    from nad.config import load_config
    return load_config(preset, [f'drive_root={DRIVE_ROOT}', f'seed={seed}', f'train_stages={train_stages}'])['run_name']

# 2단계 명령 인자. 이 칸에서 만들므로 stage1_winner 를 바꾸고 이 칸만 다시 실행하면 반영된다.
A2 = ARGS('N2', total_steps=n2_total_steps, init_from=stage1_winner, finetune_after_step=finetune_after_step)
print(ARGS('N1-pre'))
print('2단계 init_from =', repr(stage1_winner))

## 3. 원본 풀기 + [필수 확인] 즉석 변환 = 생성기 출력
STARSS23 `foa_dev.zip`, `metadata_dev.zip`에서 **main20 train 방(4, 6, 7, 12, 13, 14, 21) 녹음만** 로컬에 풉니다(val 방 9, 22와 test 방은 풀지 않고, 불러올 때도 다시 거릅니다). 수 분.
`check_match`: main20 train 클립 3개를 같은 녹음·구간·궤적으로 즉석 변환해 생성기 출력과 비교합니다(오디오 비트 단위, 라벨, 궤적). 모두 `PASS: true`여야 합니다.

In [ ]:
A = ARGS('N1-otf')
!python -B nad_run.py prepare_raw {A}
!python -B nad_run.py check_match {A}

## 4. 미리 렌더링 데이터 캐시
먼저 용량을 봅니다. 오디오 캐시(int16)는 **코랩 로컬**에 만들고(main20 train + S1 약 17GB, val/test 약 1.2GB), 라벨 캐시는 드라이브에 둡니다(작음).
zip에서 wav만 읽으므로 영상은 풀지 않습니다. 처음에는 10~30분(미측정). 끊기면 같은 셀을 다시 실행하면 이어 합니다.

In [ ]:
A = ARGS('N1-pre')
!python -B nad_run.py pre_size {A}
!df -h /content | tail -1

In [ ]:
!python -B nad_run.py prepare_pre {A}

## 5. 확인 (한 번만)
- `check_feat`: GPU 음향 특징(L, R, Mid, Side log-mel, IV, MSC)이 NTU numpy 코드와 같은지(가장자리 제외 최대 차이). 로컬 CPU에서는 log-mel 최대 차이 약 2e-4 dB, IV/MSC 약 1e-5였습니다.
- `oracle`: val 정답을 새 출력 형식으로 바꿔 후처리 → csv → 베이스라인 채점. F가 1에 가까워야 합니다(로컬에서 99.3%; 같은 클래스가 20° 안에 겹치면 합쳐져서 1보다 조금 낮음).

In [ ]:
!python -B nad_run.py check_feat {A}
!python -B nad_run.py oracle {A}

## 6. CLAP 캐시 (고정 LAION-CLAP `laion/clap-htsat-unfused`)
train은 원본 녹음 W(=Mid)에서 녹음 단위로, val/test는 클립 Mid에서 클립 단위로 2초 창을 0.5초씩 뽑습니다. 드라이브 약 0.1GB. 수십 분(미측정).

In [ ]:
!python -B nad_run.py clap {A}

## 7. 1단계 학습: 음향만 (A, B, CMC, E-2, F; 손실은 활성, sinθ, 거리)
두 실행은 **데이터만 다르고 갱신 횟수는 같습니다.**
- `N1-pre`: main20 train + S1 미리 렌더링 데이터
- `N1-otf`: STARSS23 train 방 즉석 변환(매번 새 창, 새 카메라 궤적)

모델 선택 기준은 val `F_basic`입니다(스테레오는 앞/뒤 정보가 없어 1단계는 앞/뒤를 학습하지 않음, 추론 때 모두 앞(+)으로 냄). F_ext도 매 에폭 기록됩니다.
세션이 끊기면 3, 4번을 다시 실행한 뒤 이 셀을 다시 실행하세요(이어 하기).

In [ ]:
!python -B nad_run.py train {ARGS('N1-pre')}
!python -B nad_run.py eval {ARGS('N1-pre')}

In [ ]:
!python -B nad_run.py train {ARGS('N1-otf')}
!python -B nad_run.py eval {ARGS('N1-otf')}

### 1단계 비교 (val)

In [ ]:
import json, pandas as pd
def val_row(name):
    p = f"{NAD_DIR}/runs/{name}/scores_val.json"
    if not os.path.exists(p): return None
    d = json.load(open(p)); s = d['scores']
    return dict(run=name, best_epoch=d.get('best_epoch'), F_ext=s['ext']['F'], F_basic=s['basic']['F'], DOAE_ext=s['ext']['DOA_err'],
                RDE=s['ext']['rel_dist_err'], OSA=s['ext']['onscreen_acc'])
rows = [r for r in (val_row(run_name('N1-pre')), val_row(run_name('N1-otf'))) if r]
pd.DataFrame(rows)

## 8. OWL-ViT 캐시 (고정 `google/owlvit-base-patch32`)
1) **용량을 먼저 봅니다**(드라이브 여유 공간과 비교). 2) 전처리 설정 확인(정사각형 늘리기/여백, 열 방위각). 3) 캐시 만들기: PCA(S0 train 300클립 × 10프레임) → train(+반전), val, test.
클래스 문장은 `configs/owl_prompts.yaml`(바꾸면 새 캐시 폴더). 오래 걸립니다(수 시간 가능, 미측정). 끊기면 다시 실행하면 끝난 묶음은 건너뜁니다.

In [ ]:
!python -B nad_run.py owl_size {A2}
!df -h /content/drive | tail -1

In [ ]:
!python -B nad_run.py owl_check {A2}

In [ ]:
!python -B nad_run.py owl {A2}

## 9. 2단계 학습: 전체 AV 모델 (C, D, E-1 추가, 모든 손실)
`stage1_winner`(2번 설정 칸)의 최고 모델에서 (A), (B), CMC, (E-2), (F)와 특징 통계를 가져오고, (A) 인코더는 고정합니다(`finetune_after_step` > 0이면 그 갱신부터 낮은 학습률로 같이 학습).
2단계는 미리 렌더링 데이터(영상 특징 캐시)로 학습합니다. 그다음 val로 후처리 값(τ, 열 점수 기준, ±창, 보정 켜기/끄기)을 고릅니다.

In [ ]:
assert stage1_winner, '2번 설정 칸에 stage1_winner(1단계 실행 이름)를 적으세요'
assert f'init_from="{stage1_winner}"' in A2, '2번 설정 칸을 다시 실행하세요(A2 에 stage1_winner 가 반영되지 않음)'
!python -B nad_run.py train {A2}
!python -B nad_run.py tune_post {A2}
!python -B nad_run.py eval {A2}

## 10. test 예측과 채점 (최종 보고용)
모델과 후처리는 val로 정했습니다. test는 여기서 **한 번만** 채점합니다(비공개 정답 `private_test.zip`).
비교 대상 B3(S0+S1 9200, `RESULTS_REPORT.md` 11절): test F_ext 12.85 / F_basic 17.30 / DOA_ext 39.2° / onscreen 80.1%.

In [ ]:
runs1 = [('N1-pre', run_name('N1-pre')), ('N1-otf', run_name('N1-otf'))]
for p, nm in runs1:
    if os.path.exists(f"{NAD_DIR}/runs/{nm}/best_model.pth"):
        A1 = ARGS(p)
        !python -B nad_run.py predict {A1}
        !python -B nad_run.py score_test {A1}
if stage1_winner and os.path.exists(f"{NAD_DIR}/runs/{run_name('N2')}/best_model.pth"):
    !python -B nad_run.py predict {A2}
    !python -B nad_run.py score_test {A2}

In [ ]:
def test_row(name):
    p = f"{NAD_DIR}/runs/{name}/scores_test.json"
    if not os.path.exists(p): return None
    s = json.load(open(p))['scores']
    return dict(run=name, F_ext=s['ext']['F'], F_basic=s['basic']['F'], DOAE_ext=s['ext']['DOA_err'], RDE=s['ext']['rel_dist_err'], OSA=s['ext']['onscreen_acc'])
allr = [run_name('N1-pre'), run_name('N1-otf'), run_name('N2')]
ref = dict(run='B3 S0+S1 9200 (reference)', F_ext=0.1285, F_basic=0.1730, DOAE_ext=39.2, RDE=None, OSA=0.801)
pd.DataFrame([r for r in map(test_row, allr) if r] + [ref])

## 11. W1: 손실·일반화 개선 모델 (MODEL_DEV_2.md)
1단계 `N1w`와 2단계 `N2w`를 새 이름으로 다시 학습합니다. 기존 실행과 체크포인트는 건드리지 않습니다. 시드는 1이고, 설정값은 시작값 그대로 한 번만 돌립니다.

**기존 대비 바뀌는 것**
- **손실:**
  - 방위각은 접은 각도 α(±90°)를 직접 예측합니다(기존 sinθ 대신).
  - 앞/뒤에는 라벨 스무딩 0.1을 씁니다.
  - onscreen은 소프트 라벨(τ = 5°)로 학습합니다.
  - 각 항을 정규화합니다. 거리는 ln2로, Δψ는 1.8°로 나눕니다.
- **학습:** 가중치 EMA 0.999를 씁니다(평가와 선택은 EMA로). 클리핑은 1.0입니다.
- **2단계 영상:** 영상 드롭아웃 0.2, 열 드롭아웃 0.1, (D) 입력 노이즈 0.05를 씁니다.
- **후처리:** 기존 N2의 `postproc.json`을 그대로 씁니다. θ 복원식만 바뀝니다.

**실행 순서:** 1~4번 셀을 실행한 뒤(5~10번은 건너뛰어도 됩니다) 11번 셀들을 순서대로 실행합니다. CLAP과 OWL-ViT 캐시는 기존 것을 그대로 씁니다.

In [ ]:
# W1 설정 칸. 2번 설정 칸(seed, batch_size, total_steps, n2_total_steps, finetune_after_step 등)을 그대로 씁니다.
import json, pandas as pd
W1_N1 = run_name('N1w')       # nad_N1w_otf_s1
W1_N2 = run_name('N2w')       # nad_N2w_S01_s1
OLD_N2 = run_name('N2')       # 기존 2단계 실행 (postproc.json 출처, 비교 대상)
AW1 = ARGS('N1w')
AW1_last = ARGS('N1w', eval_ckpt='last')
n2w = dict(total_steps=n2_total_steps, init_from=W1_N1, finetune_after_step=finetune_after_step, postproc_from=OLD_N2)
AW2 = ARGS('N2w', **n2w)
AW2_last = ARGS('N2w', eval_ckpt='last', **n2w)
print('1단계:', W1_N1, '| 2단계:', W1_N2, '| init_from:', W1_N1, '| 후처리:', OLD_N2, '의 postproc.json')

### 11-1. 확인 (한 번만, 학습 없음)
- **소프트 onscreen 캐시:** 드라이브 `cache/meta/train_s*_onsoft_tau5.pkl`에 만들고, 0.5로 이진화했을 때 하드 라벨과 같은 비율을 출력합니다.
  - 로컬에서 미리 만들어 두었습니다(S0 99.56%, S1 99.55%). 캐시가 이미 있으면 건너뜁니다.
- **θ 복원:** θ → (α, f) → θ가 원래 값과 같은지 봅니다(좌우 반전 포함).
- **α 형식 오라클 채점:** 99.3%가 나와야 합니다.
- **정답 입력 손실:** 정답을 예측으로 넣었을 때 ②④⑥이 0이고, ③⑤가 하한값과 같은지 봅니다.
- 이 셀은 4번(`prepare_pre`)을 마친 뒤 실행합니다.

In [ ]:
!python -B nad_run.py w1_check {AW2}
print(json.dumps(json.load(open(f"{NAD_DIR}/runs/{W1_N2}/w1_check.json")), ensure_ascii=False, indent=1))

### 11-2. 1단계 `nad_N1w_otf_s1` (즉석 변환, 손실 ①②④)
- 모델 선택은 EMA 가중치의 val `F_basic`입니다.
- 처음 200갱신 동안 10갱신마다 항별 그래디언트 노름을 `grad_norms.csv`에 기록합니다. 공유 몸통 기준이며, 가중치를 곱한 값입니다.
- 손실이 NaN이 되면 원인을 고치지 않고 멈춥니다.
- 세션이 끊기면 1~4번 셀과 11번 설정 칸을 다시 실행한 뒤 이 셀을 다시 실행하면 이어서 학습합니다.

In [ ]:
!python -B nad_run.py train {AW1}
!python -B nad_run.py eval {AW1}
!python -B nad_run.py eval {AW1_last}
display(pd.read_csv(f"{NAD_DIR}/runs/{W1_N1}/grad_norms.csv"))

### 11-3. 2단계 `nad_N2w_S01_s1`
- 11-2에서 선택된 체크포인트(EMA 기준 val F_basic 최고)로 시작합니다. 데이터(S0+S1 미리 렌더링), (A) 동결 여부, 학습률 방식은 기존 N2와 같습니다.
- W1에서 추가되는 것:
  - 손실 ①~⑥
  - onscreen 가중치 램프: 0.1에서 0.5로, 처음 5,000갱신 동안
  - 영상 드롭아웃 0.2, 열 드롭아웃 0.1, (D) 입력 노이즈 0.05
- 후처리는 새로 고르지 않고, 기존 N2의 `postproc.json`을 그대로 씁니다(`postproc_from`). 그래서 `tune_post`는 돌리지 않습니다.

In [ ]:
assert os.path.exists(f"{NAD_DIR}/runs/{W1_N1}/best_model.pth"), f'먼저 11-2({W1_N1})를 끝내세요'
assert os.path.exists(f"{NAD_DIR}/runs/{OLD_N2}/postproc.json"), f'기존 N2 의 postproc.json 이 없습니다: {OLD_N2}'
!python -B nad_run.py train {AW2}
!python -B nad_run.py eval {AW2}
!python -B nad_run.py eval {AW2_last}
display(pd.read_csv(f"{NAD_DIR}/runs/{W1_N2}/grad_norms.csv"))

### 11-4. 마지막 체크포인트, test, N2w 진단 (보고용)
- 모든 실행에서 **선택된 체크포인트(best)**와 **마지막 체크포인트(last)**를 둘 다 냅니다. last는 `--set eval_ckpt=last`로 돌리며, 결과 파일 이름에 `_last`가 붙어 기존 결과를 덮어쓰지 않습니다.
- test는 보고용입니다. 설정이나 체크포인트를 test로 고르지 않습니다.
- N2w 진단은 `diag/W1_N2w/`에 저장합니다. 영상 0 평가, 앞/뒤 정확도와 f 분포, onscreen 정밀도·재현율·OSA를 내고, 후처리는 기존 N2의 `postproc.json`(조합 a)으로 고정합니다.

In [ ]:
# 마지막 체크포인트 val 점수 (기존 N1-otf, N2 는 새 파일 scores_val_last.json 에만 쓴다. 기존 결과는 덮지 않음)
for p, extra in (('N1-otf', {}), ('N2', dict(total_steps=n2_total_steps, init_from=stage1_winner, finetune_after_step=finetune_after_step))):
    !python -B nad_run.py eval {ARGS(p, eval_ckpt='last', **extra)}
# test 예측과 채점 (최종 보고용, 선택에는 쓰지 않음): W1 은 best/last, 기존은 last 만 새로 (best 는 10번에서 이미 함)
for x in (AW1, AW1_last, AW2, AW2_last, ARGS('N1-otf', eval_ckpt='last'),
          ARGS('N2', eval_ckpt='last', total_steps=n2_total_steps, init_from=stage1_winner, finetune_after_step=finetune_after_step)):
    !python -B nad_run.py predict {x}
    !python -B nad_run.py score_test {x}
# N2w 진단 (DIAG_01 의 2~4번과 같은 표, 후처리 조합 a 고정) -> diag/W1_N2w/n2.json
D = ' '.join(f'--set {k}="{v}"' for k, v in dict(drive_root=DRIVE_ROOT, seed=seed, train_stages=train_stages, batch_size=batch_size, num_workers=num_workers).items())
!python -B diag_run.py n2 --n2 N2w --diag W1_N2w --combo a --pp_from {OLD_N2} {D}

In [ ]:
def score_row(name, split, ckpt):
    sfx = '_last' if ckpt == 'last' else ''
    p = f"{NAD_DIR}/runs/{name}/scores_{split}{sfx}.json"
    if not os.path.exists(p): return None
    d = json.load(open(p)); s = d['scores']
    return dict(run=name, split=split, ckpt=ckpt, best_epoch=d.get('best_epoch'), F_ext=s['ext']['F'], F_basic=s['basic']['F'],
                DOAE_ext=s['ext']['DOA_err'], RDE=s['ext']['rel_dist_err'], OSA=s['ext']['onscreen_acc'])
runs_cmp = [run_name('N1-otf'), run_name('N2'), W1_N1, W1_N2]
tab = pd.DataFrame([r for r in (score_row(n, s, c) for s in ('val', 'test') for n in runs_cmp for c in ('best', 'last')) if r])
print('시드', seed, '| best = val 선택 체크포인트(W1 은 EMA), last = 마지막 체크포인트(W1 은 EMA) | 후처리: N2 와 N2w 모두', OLD_N2, '의 postproc.json')
display(tab)
w = f"{NAD_DIR}/diag/W1_N2w/n2.json"
if os.path.exists(w):
    j = json.load(open(w))
    for s in ('val', 'test'):
        print(s, '영상 0:', j['item2'][s]['zeros'], '| 앞/뒤 전체:', j['item3'][s]['rows'][0], '| onscreen:', {k: j['item4'][s][k] for k in ('precision', 'recall', 'f1', 'OSA', 'OSA_all0')})

### 11-5. (선택) 1단계 방 균형 샘플링 `nad_N1w-rb_otf_s1`
**11-1~11-4 결과를 보고한 뒤에만** 실행합니다. 즉석 변환 녹음을 방별 같은 확률로, 방 안에서는 길이에 비례해 고릅니다. 1단계만 학습·평가하고, 이 실행으로 2단계는 하지 않습니다.

In [ ]:
AW1rb = ARGS('N1w-rb')
!python -B nad_run.py train {AW1rb}
!python -B nad_run.py eval {AW1rb}
!python -B nad_run.py eval {ARGS('N1w-rb', eval_ckpt='last')}
for x in (AW1rb, ARGS('N1w-rb', eval_ckpt='last')):
    !python -B nad_run.py predict {x}
    !python -B nad_run.py score_test {x}
pd.DataFrame([r for r in (score_row(run_name('N1w-rb'), s, c) for s in ('val', 'test') for c in ('best', 'last')) if r])

## 12. 1단계 ① 활성 항 세기 실험 (`N1w-a5`, `N1w-a10`)
N1w와 똑같이 학습하고, ①(활성 BCE, 전체 칸 평균)에 곱하는 상수 `w_act`만 5와 10으로 바꿉니다.
- **바뀌는 것:** `w_act` 하나뿐입니다. 이 값은 ADPIT 순열을 고르는 가중 합에도 그대로 들어갑니다.
- **그대로인 것:** ②④, EMA, 클리핑, 데이터, 시드, 20,000갱신, val `F_basic` 선택이 모두 N1w와 같습니다.
- **2단계는 돌리지 않습니다.** 결과는 `RESULT_03.md`로 보고합니다.

**실행 순서:** 1~4번 셀 → 11번 설정 칸 → 12번 설정 칸 → 12-1 → 12-2 → 12-3 → 12-4.
- 세션이 끊기면 같은 셀을 다시 실행하면 이어서 학습합니다.
- 학습 손실이 NaN이 되면 학습이 멈춥니다. 이때 다음 명령으로 넘어가지 않도록 셀이 오류를 냅니다.

In [ ]:
# 12번 설정 칸: 새 실행 이름, 그리고 N1w 와 설정이 w_act 하나만 다른지 확인 (학습 전)
import yaml, math, sys
sys.path.insert(0, NAD_DIR)
from nad.config import load_config
A_RUNS = {p: run_name(p) for p in ('N1w-a5', 'N1w-a10')}      # nad_N1w-a5_otf_s1, nad_N1w-a10_otf_s1
ref = yaml.safe_load(open(f"{NAD_DIR}/runs/{W1_N1}/config_used.yaml", encoding='utf8'))   # 실제 N1w 학습 설정
for p, rn in A_RUNS.items():
    sets = dict(drive_root=DRIVE_ROOT, seed=seed, batch_size=batch_size, total_steps=total_steps, steps_per_epoch=steps_per_epoch,
                lr=lr, num_workers=num_workers, train_stages=train_stages)
    c = dict(load_config(p, [f'{k}={v}' for k, v in sets.items()]))
    diff = {k: (ref.get(k), c.get(k)) for k in set(ref) | set(c) if ref.get(k) != c.get(k)}
    other = {k: v for k, v in diff.items() if k not in ('preset', 'run_name', 'run_dir', 'w_act')}
    print(p, rn, '| N1w 와 다른 값:', {k: diff[k] for k in diff if k not in other})
    assert not other, f'{p}: w_act 외에 N1w 와 다른 설정이 있습니다: {other}'

def run_ok(rn):
    # 학습이 끝까지 돌았는지 (NaN/발산으로 멈추면 train_summary.json 이 없다)
    if not os.path.exists(f"{NAD_DIR}/runs/{rn}/train_summary.json"):
        raise RuntimeError(f'{rn}: 학습이 끝나지 않았습니다(NaN/중단). train_out.log 를 확인하고 보고하세요. 다음 실행으로 넘어가지 않습니다.')

def gn_summary(rn):
    g = pd.read_csv(f"{NAD_DIR}/runs/{rn}/grad_norms.csv")
    m = g[(g.step >= 100) & (g.step <= 190)][['act', 'alpha', 'dist']].mean()
    return dict(run=rn, act=m['act'], alpha=m['alpha'], dist=m['dist'], alpha_over_act=m['alpha'] / m['act'])

### 12-1. `nad_N1w-a5_otf_s1` (w_act = 5)

In [ ]:
A = ARGS('N1w-a5')
!python -B nad_run.py train {A}
run_ok(A_RUNS['N1w-a5'])
!python -B nad_run.py eval {A}
!python -B nad_run.py eval {ARGS('N1w-a5', eval_ckpt='last')}
display(pd.DataFrame([gn_summary(A_RUNS['N1w-a5'])]))

### 12-2. `nad_N1w-a10_otf_s1` (w_act = 10)

In [ ]:
A = ARGS('N1w-a10')
!python -B nad_run.py train {A}
run_ok(A_RUNS['N1w-a10'])
!python -B nad_run.py eval {A}
!python -B nad_run.py eval {ARGS('N1w-a10', eval_ckpt='last')}
display(pd.DataFrame([gn_summary(A_RUNS['N1w-a10'])]))

### 12-3. test 채점 (보고용)
체크포인트는 이미 val로 골라 둔 것입니다. test로는 고르지 않습니다.

In [ ]:
for p in A_RUNS:
    run_ok(A_RUNS[p])
    for x in (ARGS(p), ARGS(p, eval_ckpt='last')):
        !python -B nad_run.py predict {x}
        !python -B nad_run.py score_test {x}

### 12-4. 요약 표 (RESULT_03.md 재료)
- N1-otf와 N1w는 다시 돌리지 않고, 기존 점수 파일(`RESULT_02.md`와 같은 값)을 읽기만 합니다.
- best는 val `F_basic`으로 고른 체크포인트이고, last는 마지막 체크포인트입니다.
- W1 계열은 둘 다 EMA 가중치입니다. N1-otf의 last는 일반 가중치입니다.

In [ ]:
def row1(rn, split, ckpt):
    p = f"{NAD_DIR}/runs/{rn}/scores_{split}{'_last' if ckpt == 'last' else ''}.json"
    if not os.path.exists(p): return None
    s = json.load(open(p))['scores']['basic']
    be = json.load(open(f"{NAD_DIR}/runs/{rn}/train_summary.json"))['best_epoch'] if ckpt == 'best' else 'last'
    return dict(run=rn, split=split, ckpt=ckpt, epoch=be, F_basic=round(100 * s['F'], 2), DOAE_basic=round(s['DOA_err'], 2))
runs12 = [run_name('N1-otf'), W1_N1] + list(A_RUNS.values())
display(pd.DataFrame([r for r in (row1(n, s, c) for s in ('val', 'test') for n in runs12 for c in ('best', 'last')) if r]))
display(pd.DataFrame([gn_summary(r) for r in [W1_N1] + list(A_RUNS.values())]).round(3))

## 13. 2단계 `N2w-a5` (1단계 = `nad_N1w-a5_otf_s1`)
N2w와 똑같이 학습합니다. 다른 점은 두 가지뿐입니다.
- **시작 가중치:** `init_from`이 N1w 대신 `nad_N1w-a5_otf_s1`의 best(EMA)입니다.
- **① 세기:** `w_act`가 5입니다.

**그대로인 것:**
- 손실 ②~⑥, EMA, 클리핑, 영상 드롭아웃, 데이터, 시드, 갱신 수, val `F_ext` 선택이 N2w와 같습니다.
- 후처리는 기존 N2의 `postproc.json`(조합 a)이고, `tune_post`는 하지 않습니다.

**실행 이름과 결과 위치:**
- 학습 run: `nad_N2w-a5_S01_s1`. 기존 N2w는 덮어쓰지 않습니다.
- 진단 결과: `diag/W1_N2w-a5/`

**실행 순서:** 1~4번 셀 → 11번 설정 칸 → 12번 설정 칸 → 13번 설정 칸 → 13-1 → 13-2 → 13-3.
- 세션이 끊기면 같은 셀을 다시 실행하면 이어서 학습합니다.

In [ ]:
# 13번 설정 칸: 실행 인자, 그리고 실제 N2w 학습 설정과 init_from·w_act 만 다른지 확인 (학습 전)
W1_N1a5 = run_name('N1w-a5')      # nad_N1w-a5_otf_s1
W1_N2a5 = run_name('N2w-a5')      # nad_N2w-a5_S01_s1
run_ok(W1_N1a5)
n2a5 = dict(total_steps=n2_total_steps, init_from=W1_N1a5, finetune_after_step=finetune_after_step, postproc_from=OLD_N2)
AW2a5 = ARGS('N2w-a5', **n2a5)
AW2a5_last = ARGS('N2w-a5', eval_ckpt='last', **n2a5)
ref2 = yaml.safe_load(open(f"{NAD_DIR}/runs/{W1_N2}/config_used.yaml", encoding='utf8'))   # 실제 N2w 학습 설정
sets2 = dict(drive_root=DRIVE_ROOT, seed=seed, batch_size=batch_size, total_steps=total_steps, steps_per_epoch=steps_per_epoch,
             lr=lr, num_workers=num_workers, train_stages=train_stages)
sets2.update(n2a5)              # ARGS 와 같은 방식: 2단계 값(total_steps 등)이 기본값을 덮는다
c2 = dict(load_config('N2w-a5', [f'{k}={v}' for k, v in sets2.items()]))
diff2 = {k: (ref2.get(k), c2.get(k)) for k in set(ref2) | set(c2) if ref2.get(k) != c2.get(k)}
other2 = {k: v for k, v in diff2.items() if k not in ('preset', 'run_name', 'run_dir', 'w_act', 'init_from')}
print(W1_N2a5, '| N2w 와 다른 값:', {k: diff2[k] for k in diff2 if k not in other2})
assert not other2, f'N2w-a5: init_from·w_act 외에 N2w 와 다른 설정이 있습니다: {other2}'

### 13-1. 학습과 val 평가

In [ ]:
!python -B nad_run.py train {AW2a5}
run_ok(W1_N2a5)
!python -B nad_run.py eval {AW2a5}
!python -B nad_run.py eval {AW2a5_last}
g = pd.read_csv(f"{NAD_DIR}/runs/{W1_N2a5}/grad_norms.csv")
display(pd.DataFrame([g[(g.step >= 100) & (g.step <= 190)].mean().rename('100~190 평균')]).round(3))

### 13-2. test 채점과 진단 (보고용)
- test 채점에는 val로 골라 둔 best 체크포인트와 last 체크포인트를 씁니다. test로 고르지 않습니다.
- 진단 항목은 N2w 진단(11-4)과 같습니다: 후처리 4조합, 영상 원래/0/섞기, 앞/뒤, onscreen.

In [ ]:
for x in (AW2a5, AW2a5_last):
    !python -B nad_run.py predict {x}
    !python -B nad_run.py score_test {x}
D = ' '.join(f'--set {k}="{v}"' for k, v in dict(drive_root=DRIVE_ROOT, seed=seed, train_stages=train_stages, batch_size=batch_size, num_workers=num_workers).items())
!python -B diag_run.py n2 --n2 N2w-a5 --diag W1_N2w-a5 --combo a --pp_from {OLD_N2} {D}

### 13-3. 요약 표 (N2, N2w, N2w-a5)

In [ ]:
def row2(rn, split, ckpt):
    p = f"{NAD_DIR}/runs/{rn}/scores_{split}{'_last' if ckpt == 'last' else ''}.json"
    if not os.path.exists(p): return None
    s = json.load(open(p))['scores']
    be = json.load(open(f"{NAD_DIR}/runs/{rn}/train_summary.json"))['best_epoch'] if ckpt == 'best' else 'last'
    return dict(run=rn, split=split, ckpt=ckpt, epoch=be, F_ext=round(100 * s['ext']['F'], 2), F_basic=round(100 * s['basic']['F'], 2),
                DOAE_ext=round(s['ext']['DOA_err'], 2), DOAE_basic=round(s['basic']['DOA_err'], 2), RDE=round(s['ext']['rel_dist_err'], 4),
                OSA=round(100 * s['ext']['onscreen_acc'], 2))
runs13 = [OLD_N2, W1_N2, W1_N2a5]
display(pd.DataFrame([r for r in (row2(n, s, c) for s in ('val', 'test') for n in runs13 for c in ('best', 'last')) if r]))
for tag in ('W1_N2w', 'W1_N2w-a5'):
    w = f"{NAD_DIR}/diag/{tag}/n2.json"
    if os.path.exists(w):
        j = json.load(open(w))
        for s in ('val', 'test'):
            print(tag, s, '| 영상 0:', {k: round(v, 4) for k, v in j['item2'][s]['zeros'].items() if k != 'snap'},
                  '| 앞/뒤 전체:', round(j['item3'][s]['rows'][0]['acc'], 4),
                  '| onscreen:', {k: round(j['item4'][s][k], 4) for k in ('precision', 'recall', 'f1', 'OSA', 'OSA_all0')})